# 02 — Baseline

**This notebook does not train anything.** The baseline was produced by

```bash
make baseline    # uv run python -m src.train --model lgbm --notes "baseline"
```

so that it lands in `experiments.csv` with a fold fingerprint and a saved
`oof_*.npy`. Training in a notebook cell instead would produce a number nobody
can reproduce and predictions that cannot be blended. This notebook *reads those
artifacts and says what they mean*.

What a baseline is for: it is the reference every later change is measured
against, so it deliberately uses the 19 raw features with no engineering. Its
job is not to be good, it is to be **honest and repeatable**.

Sections: integrity checks → how good is it → CV vs LB → what it predicts →
what it learned → where it is wrong → what to do next.

In [ ]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, roc_curve

from src import config as C
from src import cv as CV
from src.data import load
from src.utils import LOG_FILE, savefig

sns.set_theme(style="whitegrid")

log = pd.read_csv(LOG_FILE)
log

## 1. Integrity checks

Four things must hold before any number in this notebook means anything. Each
one has a specific failure it is guarding against.

In [ ]:
# the baseline = the earliest lgbm run in the log
run = log[log.model == "lgbm"].iloc[0]
run_id = run.run_id
print(f"run_id  {run_id}\nnotes   {run.notes!r}\nlogged  cv={run.cv_score}  lb={run.public_lb}")

train, test = load()
y = train[C.TARGET].values
oof = np.load(C.EXPERIMENTS / f"oof_{run_id}.npy")
test_pred = np.load(C.EXPERIMENTS / f"test_{run_id}.npy")

In [ ]:
now = CV.fold_fingerprint(train)
checks = {
    # a stale fingerprint means this oof was made under a different fold scheme;
    # it is the same shape and dtype either way, so nothing else would catch it
    "fold fingerprint matches config": (run.folds, now, run.folds == now),
    # NaN marks "never validated" — stratified folds partition, so there are none
    "every row validated (no NaN)": (int(np.isnan(oof).sum()), 0, not np.isnan(oof).any()),
    "oof length == len(train)": (len(oof), len(train), len(oof) == len(train)),
    "test length == len(test)": (len(test_pred), len(test), len(test_pred) == len(test)),
}
for name, (got, want, ok) in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name:34s} got={got}  want={want}")
assert all(ok for *_, ok in checks.values())

# and the logged score must be reproducible from the saved array
recomputed = CV.score(y, oof)
print(
    f"\nlogged cv_score {run.cv_score:.6f}  recomputed from oof {recomputed:.6f}  "
    f"diff {abs(recomputed - run.cv_score):.2e}"
)

## 2. How good is it?

The pooled OOF score is one number; the five fold scores are what tell you how
much to trust it.

In [ ]:
folds = CV.get_folds(train)
per_fold = pd.DataFrame(
    [
        {"fold": k, "n_valid": len(va), "auc": roc_auc_score(y[va], oof[va])}
        for k, (_, va) in enumerate(folds)
    ]
).set_index("fold")
per_fold.loc["pooled"] = [len(oof), recomputed]
per_fold["n_valid"] = per_fold.n_valid.astype(int)
print(per_fold.to_string(float_format=lambda v: f"{v:.6f}"))

f = per_fold.auc[:-1]
print(f"\nfold SD {f.std(ddof=1):.6f}   range {f.max() - f.min():.6f}")
print(f"base rate {y.mean():.4f}  ->  a model predicting the base rate scores 0.5")

The fold SD is the practical yardstick. It is **larger** than the ±0.00085
pure-evaluation noise measured in `learnings.md`, because fold-to-fold variation
also includes training on different data — the bootstrap there holds the fitted
model fixed and only resamples the rows being scored.

For deciding whether the *next* run is better, neither number is the right one:
runs share identical folds, so comparisons are paired and roughly 16× more
precise than either absolute score. See **Judging whether a CV difference is
real** in `learnings.md`. The cheap test is whether the per-fold differences
agree in sign.

## 3. CV vs the public leaderboard

The EDA concluded that OOF and test are drawn from the same distribution, so CV
should track the LB closely and *a gap would indicate a bug, not a shift*. That
prediction is now testable.

In [ ]:
gap = run.cv_score - run.public_lb
print(f"CV {run.cv_score:.6f}   public LB {run.public_lb:.6f}   gap {gap:+.6f}")


# Hanley-McNeil SE of an AUC. Conservative by ~1.4x vs the bootstrap measured
# in learnings.md, so divide by that to get a realistic figure.
def auc_se(a, n_pos, n_neg):
    q1, q2 = a / (2 - a), 2 * a**2 / (1 + a)
    return np.sqrt(
        (a * (1 - a) + (n_pos - 1) * (q1 - a**2) + (n_neg - 1) * (q2 - a**2)) / (n_pos * n_neg)
    )


# the public split share is not published, so bracket it
print("\nsampling noise on the public LB, by unknown split share:")
for share in (0.10, 0.20, 0.50):
    n = int(len(test) * share)
    n_pos = int(n * y.mean())
    se = auc_se(run.public_lb, n_pos, n - n_pos) / 1.4  # bootstrap-calibrated
    flag = "gap is inside this" if abs(gap) < 2 * se else "gap EXCEEDS this"
    print(f"  public = {share:>4.0%} of test ({n:>7,} rows):  +/-2SE = {2 * se:.5f}   <- {flag}")

The gap sits inside ordinary sampling noise if the public set is 10-20% of test,
which is the usual range for a Playground Series competition. It would exceed
2 SE only if the public set were as large as half the test data. The share is not
published, so read this as *consistent with noise*, not as proof — what actually
settles it is whether CV and LB keep moving together across several runs.

Two things to keep in mind as more runs land:

- A **constant offset** between CV and LB is harmless. What would matter is the
  two moving in *opposite* directions — that is the signature of overfitting the
  validation scheme. `uv run python -m src.log_lb` with no arguments prints the
  rank correlation, and says it needs at least three scored runs to mean
  anything.
- There is a known small optimism in the OOF: `train.py` early-stops on the same
  rows it then scores. Measured on fold 0 it is **+0.000126** — about 5% of the
  gap, so not the explanation. It is systematic and always positive, so absolute
  OOF slightly overstates performance, but it applies equally to every run and
  leaves run-to-run comparisons valid.

## 4. What the model actually predicts

AUC only grades *ranking*. Two models with identical AUC can produce completely
differently-scaled probabilities, and if the eventual use is "call the customers
above 60% risk" then the scale matters as much as the order.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# (a) separation
for cls, name in [(0, "stayed"), (1, "churned")]:
    sns.kdeplot(oof[y == cls], ax=axes[0], fill=True, alpha=0.4, label=name, clip=(0, 1))
axes[0].set_title("OOF predicted probability by outcome")
axes[0].set_xlabel("predicted P(churn)")
axes[0].legend()

# (b) ROC
fpr, tpr, _ = roc_curve(y, oof)
axes[1].plot(fpr, tpr, lw=2, label=f"AUC {recomputed:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="chance")
axes[1].set(xlabel="false positive rate", ylabel="true positive rate", title="ROC (OOF)")
axes[1].legend()

# (c) calibration
prob_true, prob_pred = calibration_curve(y, oof, n_bins=20, strategy="quantile")
axes[2].plot(prob_pred, prob_true, marker="o", ms=4, lw=1.5)
axes[2].plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
axes[2].set(xlabel="mean predicted", ylabel="observed rate", title="Calibration (20 quantile bins)")
axes[2].legend()

fig.tight_layout()
savefig(fig, "02_baseline_predictions")

In [ ]:
print(
    f"mean predicted {oof.mean():.6f}   observed base rate {y.mean():.6f}   "
    f"bias {oof.mean() - y.mean():+.6f}"
)
print(f"test-set mean predicted {test_pred.mean():.6f}   (should sit near the base rate)")
print(f"\nmax |calibration error| across bins: {np.abs(prob_true - prob_pred).max():.4f}")

# what a business threshold would actually buy you
print("\nif you called the top-k% highest-risk customers:")
order = np.argsort(-oof)
for k in (0.05, 0.10, 0.20, 0.30):
    top = order[: int(k * len(oof))]
    print(
        f"  top {k:>4.0%}: precision {y[top].mean():.3f}   "
        f"recall {y[top].sum() / y.sum():.3f}   lift {y[top].mean() / y.mean():.2f}x"
    )

LightGBM's `binary` objective optimises log loss, so the probabilities come out
close to calibrated without a post-hoc step — worth confirming rather than
assuming, since the metric would not have told us either way.

The lift table is the version of this a business reader understands, and it
belongs in the final writeup: it converts "AUC 0.916" into "call the riskiest
10% and roughly *N*× as many of them churn as in the base population".

## 5. What the model learned

Compare gain-based importance against the univariate AUC ranking from notebook
01. They answer different questions, and the difference is the interesting part.

In [ ]:
imp = pd.read_csv(C.EXPERIMENTS / f"importance_{run_id}.csv")
imp["gain_share"] = imp.gain / imp.gain.sum()

# univariate AUC, recomputed here so the two rankings sit side by side
uni = []
for c in imp.feature:
    s = (
        train[c]
        if pd.api.types.is_numeric_dtype(train[c])
        else train[c].map(pd.Series(y, index=train.index).groupby(train[c]).mean())
    )
    a = roc_auc_score(y, s)
    uni.append({"feature": c, "univariate_auc": max(a, 1 - a)})

cmp = (
    imp.merge(pd.DataFrame(uni), on="feature")
    .assign(
        gain_rank=lambda d: d.gain.rank(ascending=False).astype(int),
        uni_rank=lambda d: d.univariate_auc.rank(ascending=False).astype(int),
    )
    .assign(rank_shift=lambda d: d.uni_rank - d.gain_rank)
    .sort_values("gain", ascending=False)
)
print(
    cmp[
        ["feature", "gain_share", "gain_rank", "univariate_auc", "uni_rank", "rank_shift"]
    ].to_string(index=False, float_format=lambda v: f"{v:.4f}")
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top = cmp.head(12)
sns.barplot(data=top, x="gain_share", y="feature", ax=axes[0])
axes[0].set(title="Gain share in the model (multivariate)", xlabel="share of total gain")
sns.barplot(
    data=cmp.sort_values("univariate_auc", ascending=False).head(12),
    x="univariate_auc",
    y="feature",
    ax=axes[1],
)
axes[1].axvline(0.5, color="k", ls="--", lw=1)
axes[1].set(title="Univariate AUC (marginal)", xlabel="AUC alone")
fig.tight_layout()
savefig(fig, "02_importance_vs_univariate")

`rank_shift > 0` means a feature is *more* useful to the model than it looked
alone; `< 0` means the opposite — its marginal signal was already covered by
something else.

The headline case is `tenure`: the strongest single feature in the EDA, but
demoted once `contract` is in the model, because contract length already encodes
much of what tenure was carrying. This is the redundancy effect written up in
`learnings.md` — **univariate AUC and gain answer different questions, and a
feature that "looks strong" alone can add little on top of the others.**

The practical consequence for feature engineering: build features that add
something the existing columns do *not* already say. A ratio between two
existing columns often does; a rescaling of one of them does not.

## 6. Where is it wrong?

Pooled AUC hides the segments. Two things are worth separating:

- **Ranking quality within a segment** (AUC computed inside the group) — is the
  model just as discriminating for month-to-month customers as for two-year ones?
- **Calibration within a segment** (mean predicted vs. observed rate) — is it
  systematically over- or under-stating risk for a group?

A segment can rank perfectly and still be badly biased, and vice versa.

In [ ]:
def segment_report(df: pd.DataFrame, col: str) -> pd.DataFrame:
    rows = []
    for level, idx in df.groupby(col, observed=True).indices.items():
        yi, pi = y[idx], oof[idx]
        rows.append(
            {
                "segment": f"{col}={level}",
                "n": len(idx),
                "observed": yi.mean(),
                "predicted": pi.mean(),
                "bias": pi.mean() - yi.mean(),
                # AUC is undefined inside a segment that is all one class
                "auc_within": roc_auc_score(yi, pi) if 0 < yi.mean() < 1 else np.nan,
            }
        )
    return pd.DataFrame(rows)


seg = pd.concat(
    [
        segment_report(train, c)
        for c in ["contract", "internetservice", "paymentmethod", "seniorcitizen"]
    ]
).sort_values("auc_within")
print(seg.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

In [ ]:
# tenure bucketed at the contract boundaries the EDA flagged (12 and 24 months)
# left as an ordered Categorical so the buckets stay in numeric order
train_t = train.assign(tenure_bucket=pd.cut(train.tenure, [0, 6, 12, 24, 48, 72]))
tenure_seg = segment_report(train_t, "tenure_bucket")
print(tenure_seg.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=seg.dropna(subset=["auc_within"]), x="auc_within", y="segment", ax=axes[0])
axes[0].axvline(recomputed, color="crimson", ls="--", lw=1, label="pooled OOF")
axes[0].set(title="AUC within segment", xlabel="AUC")
axes[0].legend()
sns.barplot(data=tenure_seg, x="bias", y="segment", ax=axes[1])
axes[1].axvline(0, color="k", lw=1)
axes[1].set(title="Calibration bias by tenure (predicted - observed)", xlabel="bias")
fig.tight_layout()
savefig(fig, "02_segment_errors")

Read the within-segment AUCs against the pooled score with care: **a segment's
AUC is not comparable to the pooled one.** Conditioning on a strong feature
removes that feature's contribution from the ranking problem, so every segment
of a strong predictor will score *lower* than the pooled model even when nothing
is wrong. What is diagnostic is the comparison *between* segments of the same
column, and the bias column, which has no such artefact.

## 7. What this licenses, and what is next

**Established.** A reproducible 0.9163 OOF on raw features, folds frozen and
fingerprinted, artifacts saved, a valid submission on the board at 0.9139, and a
CV-LB relationship consistent with the EDA's prediction. Everything from here is
measured against this run.

**Not established.** That any particular feature helps; that LightGBM is the
right model; that the hyperparameters are near-optimal. None of that has been
tested yet, and the baseline deliberately does not try.

**Next, in the order the README ranks value per hour:**

1. **Features that encode structure the raw columns do not already carry.**
   From notebook 01, and chosen to be non-redundant with `contract`:
   - `totalcharges / (tenure * monthlycharges)` — the residual against the
     implied bill. Centred on 1.0 with a long tail, so it captures plan changes
     and one-off charges that no single input states.
   - `totalcharges / tenure` — realised average spend vs. the current rate.
   - count of subscribed add-ons (the six service columns as a 0-6 integer).
   - `tenure` bucketed at the contract boundaries (12, 24 months).
2. **A structurally different model for blend material** — CatBoost is already
   wired up. Per the README, a second booster on the same features correlates
   ~0.9 and blends to almost nothing; the value comes from a different estimator
   shape, so judge it on the correlation matrix `src.blend` prints, not on its
   solo score.
3. **Hyperparameters last**, and worth less than expected.

One change per run, always with `--notes`, and record the reasoning in
`experiments.md` — the CSV holds the numbers, that file holds the *why*.

```bash
uv run python -m src.train --model lgbm --notes "add charges ratios + addon count"
```